In [1]:
pip install torch torchvision matplotlib scikit-learn pillow

Note: you may need to restart the kernel to use updated packages.


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

import matplotlib.pyplot as plt
import numpy as np

In [3]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("Using device:", device)

Using device: mps


In [4]:
TRAIN_DIR = "data/seg_train/seg_train"
TEST_DIR = "data/seg_test/seg_test"

BATCH_SIZE = 32

### define image transformations

In [5]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

### Load dataset

In [6]:
train_dataset = datasets.ImageFolder(
    TRAIN_DIR,
    transform=train_transform
)

test_dataset = datasets.ImageFolder(
    TEST_DIR,
    transform=test_transform
)

class_names = train_dataset.classes

print(class_names)
print("Training images:", len(train_dataset))
print("Test images:", len(test_dataset))

['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street']
Training images: 14034
Test images: 3000


### create dataloaders

In [7]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

### load pretrained ResNet18 :
### This is the main transfer-learning step.
### The model already has weights learned from ImageNet.


In [8]:
model = models.resnet18(
    weights=models.ResNet18_Weights.DEFAULT
)

9.0%

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /Users/siddhiwanzkhade/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100.0%


### freeze pretrained layers

In [9]:
for param in model.parameters():
    param.requires_grad = False

In [10]:
#replace final layer cause resnet pretrained layer has 100 classes and i only have 6 classes
num_features = model.fc.in_features

model.fc = nn.Linear(
    num_features,
    len(class_names)
)

model = model.to(device)

In [13]:
#define loss and optimizer
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.fc.parameters(),
    lr=0.001
)

### Traning (Only new final layer is being trained)

In [14]:
EPOCHS = 10

for epoch in range(EPOCHS):

    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / len(train_loader)
    epoch_accuracy = 100 * correct / total

    print(
        f"Epoch {epoch+1}/{EPOCHS} | "
        f"Loss: {epoch_loss:.4f} | "
        f"Accuracy: {epoch_accuracy:.2f}%"
    )

Epoch 1/10 | Loss: 0.5816 | Accuracy: 80.79%
Epoch 2/10 | Loss: 0.3608 | Accuracy: 87.42%
Epoch 3/10 | Loss: 0.3353 | Accuracy: 88.09%
Epoch 4/10 | Loss: 0.3172 | Accuracy: 88.53%
Epoch 5/10 | Loss: 0.3057 | Accuracy: 89.23%
Epoch 6/10 | Loss: 0.3103 | Accuracy: 88.62%
Epoch 7/10 | Loss: 0.3104 | Accuracy: 88.64%
Epoch 8/10 | Loss: 0.3010 | Accuracy: 88.86%
Epoch 9/10 | Loss: 0.3034 | Accuracy: 88.96%
Epoch 10/10 | Loss: 0.2937 | Accuracy: 89.35%


### Model evaluation

In [15]:
model.eval()

test_loss = 0.0
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        test_loss += loss.item()

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

test_loss /= len(test_loader)
test_accuracy = 100 * correct / total

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.2f}%")

Test Loss: 0.2817
Test Accuracy: 89.60%


In [16]:
torch.save(
    model.state_dict(),
    "resnet18_transfer_learning.pth"
)

print("Model saved successfully.")

Model saved successfully.


In [17]:
from torchvision import models
import torch.nn as nn
import torch

loaded_model = models.resnet18(weights=None)

num_features = loaded_model.fc.in_features
loaded_model.fc = nn.Linear(num_features, 6)

loaded_model.load_state_dict(
    torch.load(
        "resnet18_transfer_learning.pth",
        map_location=device
    )
)

loaded_model = loaded_model.to(device)
loaded_model.eval()

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_sta